# 1. Scrape LinkedIn job postings

Selenium scraper for LinkedIn's public job search. Collects title, company, location, salary, full JD, seniority, employment type, job function and industries; de-duplicates against existing rows and autosaves every 20 jobs. Output: `linkedin_jobs_english.csv` (1,402 postings in our run), which notebook 04 loads into the Company / Position tables.

In [ ]:
!pip install selenium

In [ ]:
import os
import time
import csv
import random
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException

# ==========================================
# LinkedIn Jobs Scraper (guest job search, no login)
# ==========================================

TARGET_JOBS_COUNT = 1000
SAVE_BATCH_SIZE = 20  
URL = "https://www.linkedin.com/jobs/search?trk=guest_homepage-basic_guest_nav_menu_jobs&position=1&pageNum=0"
OUTPUT_FILE = "linkedin_jobs_english.csv"

# CSV columns
FIELD_NAMES = [
    "Job Title", "Company", "Location", "Salary Range",
    "Detailed JD", "Seniority Level", "Employment Type",
    "Job Function", "Industries"
]

def setup_driver():
    options = Options()
    options.add_argument("--lang=en-US")
    prefs = {
        "intl.accept_languages": "en-US,en",
        "profile.default_content_setting_values.notifications": 2
    }
    options.add_experimental_option("prefs", prefs)
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)

    driver = webdriver.Chrome(options=options)
    driver.execute_cdp_cmd('Network.setUserAgentOverride', {"userAgent": 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'})
    return driver

def random_sleep(min_s=1, max_s=3):
    time.sleep(random.uniform(min_s, max_s))

def close_login_popup(driver):
    dismiss_selectors = [
        "button[aria-label='Dismiss']",
        "button.contextual-sign-in-modal__modal-dismiss",
        "button.modal__dismiss",
        "icon>svg" 
    ]
    for selector in dismiss_selectors:
        try:
            elements = driver.find_elements(By.CSS_SELECTOR, selector)
            for el in elements:
                if el.is_displayed():
                    driver.execute_script("arguments[0].click();", el)
                    print("Login popup detected and closed.")
                    time.sleep(1.5)
                    return True 
        except Exception:
            continue
    return False

def load_existing_records(file_path):
    existing_jobs = set()
    if os.path.exists(file_path):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                reader = csv.DictReader(f)
                for row in reader:
                    title = row.get("Job Title", "").strip()
                    company = row.get("Company", "").strip()
                    loc = row.get("Location", "").strip()
                    if title or company:
                        existing_jobs.add((title, company, loc))
            print(f"Loaded {len(existing_jobs)} existing records for de-duplication.")
        except Exception as e:
            print(f"Failed to read existing file: {e}")
    return existing_jobs

def save_batch(data_buffer, file_path, is_first_write):
    if not data_buffer: return is_first_write
    mode = 'w' if is_first_write else 'a'
    try:
        with open(file_path, mode, newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=FIELD_NAMES)
            if is_first_write: writer.writeheader()
            writer.writerows(data_buffer)
        print(f"[autosave] appended {len(data_buffer)} rows.")
    except Exception as e:
        print(f"Save failed: {e}")
    return False 

def scrape_linkedin_jobs():
    existing_jobs = load_existing_records(OUTPUT_FILE)
    is_first_write = not os.path.exists(OUTPUT_FILE)
    
    driver = setup_driver()
    driver.get(URL)
    driver.maximize_window()
    
    print("Browser opened. If a CAPTCHA appears, solve it manually...")
    time.sleep(5) 
    
    # --- Phase 1: scroll to load the job list ---
    print(f"\nScrolling to load {TARGET_JOBS_COUNT} jobs...")
    loaded_count = 0
    attempts_without_new_jobs = 0
    MAX_ATTEMPTS = 5 
    
    while loaded_count < TARGET_JOBS_COUNT:
        close_login_popup(driver)
        job_cards = driver.find_elements(By.CSS_SELECTOR, "ul.jobs-search__results-list > li")
        current_count = len(job_cards)
        
        if current_count > 0:
            try: driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", job_cards[-1])
            except: pass
            
        time.sleep(2) 
        
        try:
            see_more_btn = driver.find_element(By.CSS_SELECTOR, "button.infinite-scroller__show-more-button")
            if see_more_btn.is_displayed():
                close_login_popup(driver)
                driver.execute_script("arguments[0].click();", see_more_btn)
                print("-> clicked 'See more jobs', waiting...")
                try:
                    WebDriverWait(driver, 10).until(
                        lambda d: len(d.find_elements(By.CSS_SELECTOR, "ul.jobs-search__results-list > li")) > current_count
                    )
                except TimeoutException: pass
        except NoSuchElementException: pass
            
        new_cards = driver.find_elements(By.CSS_SELECTOR, "ul.jobs-search__results-list > li")
        new_count = len(new_cards)
        
        if new_count == loaded_count:
            attempts_without_new_jobs += 1
            print(f"-> no new jobs loaded ({attempts_without_new_jobs}/{MAX_ATTEMPTS})")
            driver.execute_script("window.scrollBy(0, -500);")
            time.sleep(1)
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)
            if attempts_without_new_jobs >= MAX_ATTEMPTS: break
        else:
            attempts_without_new_jobs = 0 
            print(f"Jobs loaded so far: {new_count}")
            
        loaded_count = new_count
        
    print(f"\nJob list loaded, scraping detail panes...")
    
    # --- Phase 2: extract details ---
    job_cards = driver.find_elements(By.CSS_SELECTOR, "ul.jobs-search__results-list > li")
    
    last_title = ""
    last_company = ""
    data_buffer = [] 
    scraped_in_this_session = 0
    
    for index in range(min(TARGET_JOBS_COUNT, len(job_cards))):
        close_login_popup(driver)
        try:
            card = driver.find_elements(By.CSS_SELECTOR, "ul.jobs-search__results-list > li")[index]
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", card)
            random_sleep(1, 1.5)
            
            try:
                card_link = card.find_element(By.CSS_SELECTOR, "a")
                driver.execute_script("arguments[0].click();", card_link)
            except:
                driver.execute_script("arguments[0].click();", card)
            
            random_sleep(2.5, 4.0) 
            
            try: right_pane = driver.find_element(By.CSS_SELECTOR, ".two-pane-serp-page__detail-view")
            except NoSuchElementException: continue
                
            job_title = ""
            company = ""
            location = ""
            
            # --- job title / company / location from the detail pane ---
            try: job_title = right_pane.find_element(By.CSS_SELECTOR, "h2.topcard__title, h2.top-card-layout__title").text.strip()
            except: pass
            
            try: company = right_pane.find_element(By.CSS_SELECTOR, "a.topcard__org-name-link").text.strip()
            except: pass
            
            try: location = right_pane.find_element(By.CSS_SELECTOR, "span.topcard__flavor--bullet").text.strip()
            except: pass
            
            # the pane may not have refreshed yet; retry once if it still shows the previous job
            if job_title == last_title and company == last_company:
                time.sleep(3) 
                try: job_title = right_pane.find_element(By.CSS_SELECTOR, "h2.topcard__title, h2.top-card-layout__title").text.strip()
                except: pass
                try: company = right_pane.find_element(By.CSS_SELECTOR, "a.topcard__org-name-link").text.strip()
                except: pass
                try: location = right_pane.find_element(By.CSS_SELECTOR, "span.topcard__flavor--bullet").text.strip()
                except: pass

            last_title = job_title
            last_company = company
            
            job_key = (job_title, company, location)
            if job_key in existing_jobs:
                print(f"[skip] already scraped: {job_title} @ {company}")
                continue
            
            # --- build the row ---
            job_dict = {
                "Job Title": job_title, "Company": company, "Location": location, 
                "Salary Range": "", "Detailed JD": "", "Seniority Level": "", 
                "Employment Type": "", "Job Function": "", "Industries": ""
            }
            
            try: job_dict["Salary Range"] = right_pane.find_element(By.CSS_SELECTOR, ".salary-compensation-route-estimate").text.strip()
            except: pass
            
            try:
                show_more_btn = right_pane.find_element(By.CSS_SELECTOR, "button.show-more-less-html__button")
                driver.execute_script("arguments[0].click();", show_more_btn)
                time.sleep(1)
            except: pass
            
            try: job_dict["Detailed JD"] = right_pane.find_element(By.CSS_SELECTOR, "div.show-more-less-html__markup").text.strip()
            except: pass
            
            criteria_items = right_pane.find_elements(By.CSS_SELECTOR, "ul.description__job-criteria-list li")
            for item in criteria_items:
                try:
                    header = item.find_element(By.CSS_SELECTOR, "h3").text.strip().lower()
                    value = item.find_element(By.CSS_SELECTOR, "span").text.strip()
                    if "seniority" in header: job_dict["Seniority Level"] = value
                    elif "employment" in header: job_dict["Employment Type"] = value
                    elif "function" in header: job_dict["Job Function"] = value
                    elif "industries" in header: job_dict["Industries"] = value
                except: continue
                    
            data_buffer.append(job_dict)
            existing_jobs.add(job_key) 
            scraped_in_this_session += 1
            print(f"[new] {job_title} @ {company} - {location}")
            
            if len(data_buffer) >= SAVE_BATCH_SIZE:
                is_first_write = save_batch(data_buffer, OUTPUT_FILE, is_first_write)
                data_buffer.clear()
            
        except Exception as e:
            print(f"Error on job #{index + 1}: {e}")
            continue

    driver.quit()
    if data_buffer: save_batch(data_buffer, OUTPUT_FILE, is_first_write)
    print(f"\nDone. {scraped_in_this_session} new jobs scraped this run.")

if __name__ == '__main__':
    scrape_linkedin_jobs()